In [1]:
import os
from kaggle_secrets import UserSecretsClient


secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

print("W&B key loaded!")

W&B key loaded!


In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("All imports done!")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

All imports done!


In [3]:
BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'


wandb.init(project="23f3003600-t12026", name="model1-cnn")

print("Paths and W&B ready!")

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: setting up run h5s56ziu
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260318_121229-h5s56ziu
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-cnn
wandb: ⭐️ View project at https://wandb.ai/23f3003600s-indian-institute-of-technology-madras/23f3003600-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003600s-indian-institute-of-technology-madras/23f3003600-t12026/runs/h5s56ziu


Paths and W&B ready!


In [4]:
# Collect all audio files with their genre labels
data = []
genres_path = os.path.join(BASE, 'genres_stems')

for genre in os.listdir(genres_path):
    genre_folder = os.path.join(genres_path, genre)
    if not os.path.isdir(genre_folder):
        continue
    for song_folder in os.listdir(genre_folder):
        song_path = os.path.join(genre_folder, song_folder)
        audio_file = os.path.join(song_path, 'other.wav')
        if os.path.exists(audio_file):
            data.append((audio_file, genre))

train_df = pd.DataFrame(data, columns=['filepath', 'genre'])

print(f"Total files found: {len(train_df)}")
print(train_df['genre'].value_counts())

Total files found: 1000
genre
disco        100
metal        100
reggae       100
blues        100
rock         100
classical    100
jazz         100
hiphop       100
country      100
pop          100
Name: count, dtype: int64


In [5]:
# Convert genre names to numbers
# CNN needs numbers not strings
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['genre'])

print("Genre to number mapping:")
for i, genre in enumerate(le.classes_):
    print(f"  {genre} → {i}")

NUM_CLASSES = len(le.classes_)
print(f"\nTotal genres: {NUM_CLASSES}")

Genre to number mapping:
  blues → 0
  classical → 1
  country → 2
  disco → 3
  hiphop → 4
  jazz → 5
  metal → 6
  pop → 7
  reggae → 8
  rock → 9

Total genres: 10


In [6]:
def get_melspec(file_path, sr=22050, duration=30, n_mels=64, n_fft=2048, hop=512):
   
    try:
        
        y, sr = librosa.load(file_path, sr=sr, duration=duration)
        
        
        target_len = sr * duration
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]
        
        
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                              n_fft=n_fft, hop_length=hop)
        
        
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
       
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        
        return mel_db.astype(np.float32)
    
    except:
       
        return np.zeros((n_mels, 1292), dtype=np.float32)



sample_mel = get_melspec(train_df['filepath'][0])
print(f"Mel-spectrogram shape: {sample_mel.shape}")


Mel-spectrogram shape: (64, 1292)


In [7]:
class AudioDataset(Dataset):
   
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
    
    def __len__(self):
        
        return len(self.filepaths)
    
    def __getitem__(self, idx):
        
        mel = get_melspec(self.filepaths[idx])
        
        
        mel = torch.tensor(mel).unsqueeze(0)
        
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

In [8]:

X_train, X_val, y_train, y_val = train_test_split(
    train_df['filepath'].values,
    train_df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'].values
)


train_dataset = AudioDataset(X_train, y_train)
val_dataset = AudioDataset(X_val, y_val)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 50
Val batches: 13


In [9]:
class SimpleCNN(nn.Module):
    """
    A simple CNN with 3 convolutional layers.
    
    Conv layer = finds patterns in the spectrogram image
    MaxPool = shrinks the image (keeps important info)
    Linear = final classification layer
    """
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
       
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        
        self.fc = nn.Linear(64, num_classes)
    
    def forward(self, x):
        
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.global_pool(x)          
        x = x.view(x.size(0), -1)        
        x = self.fc(x)                   
        return x



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
print(model)

Using device: cpu
SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)


In [10]:
#loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 10

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    
    
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for mels, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        mels = mels.to(device)
        labels = labels.to(device)
        
        
        outputs = model(mels)
        
        
        loss = criterion(outputs, labels)
        
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
    
    train_acc = train_correct / train_total
    
    
    model.eval()
    val_preds_all = []
    val_labels_all = []
    
    with torch.no_grad():
        for mels, labels in val_loader:
            mels = mels.to(device)
            outputs = model(mels)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds_all.extend(preds)
            val_labels_all.extend(labels.numpy())
    
    val_f1 = f1_score(val_labels_all, val_preds_all, average='macro')
    
    
    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss / len(train_loader),
        "train_acc": train_acc,
        "val_f1": val_f1
    })
    
    print(f"Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f} | Train Acc={train_acc:.4f} | Val F1={val_f1:.4f}")

print("Training complete!")

Starting training...


Epoch 1/10: 100%|██████████| 50/50 [02:58<00:00,  3.56s/it]


Epoch 1: Loss=2.2975 | Train Acc=0.1375 | Val F1=0.1207


Epoch 2/10: 100%|██████████| 50/50 [02:15<00:00,  2.71s/it]


Epoch 2: Loss=2.2104 | Train Acc=0.1950 | Val F1=0.1301


...

Epoch 8: Loss=2.0193 | Train Acc=0.2550 | Val F1=0.1637


Epoch 9/10: 100%|██████████| 50/50 [02:15<00:00,  2.70s/it]


Epoch 9: Loss=1.9964 | Train Acc=0.2612 | Val F1=0.1485


Epoch 10/10: 100%|██████████| 50/50 [02:16<00:00,  2.73s/it]


Epoch 10: Loss=1.9662 | Train Acc=0.2687 | Val F1=0.2243
Training complete!


In [11]:

test_df = pd.read_csv(os.path.join(BASE, 'test.csv'))

model.eval()
test_preds = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        file_path = os.path.join(BASE, row['filename'])
        mel = get_melspec(file_path)
        mel_tensor = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(device)
        output = model(mel_tensor)
        pred = output.argmax(dim=1).item()
        test_preds.append(pred)


test_genres = le.inverse_transform(test_preds)

submission = pd.DataFrame({
    'id': test_df['id'],
    'genre': test_genres
})

submission.to_csv('submission.csv', index=False)
print("Submission saved!")
print(submission['genre'].value_counts())


torch.save(model.state_dict(), 'cnn_model.pth')
print("Model saved!")

wandb.finish()

wandb: uploading data; updating run metadata


Submission saved!
genre
classical    1504
metal         908
blues         232
hiphop        147
reggae        121
disco          47
country        40
rock           21
Name: count, dtype: int64
Model saved!


wandb: uploading data
wandb: uploading summary, console lines 66-77
wandb: 
wandb: Run history:
wandb:      epoch ▁▂▃▃▄▅▆▆▇█
wandb:  train_acc ▁▄▅▆▆▆▆▇██
wandb: train_loss █▆▄▃▃▃▂▂▂▁
wandb:     val_f1 ▁▂▅▃▆▃▅▄▃█
wandb: 
wandb: Run summary:
wandb:      epoch 10
wandb:  train_acc 0.26875
wandb: train_loss 1.96623
wandb:     val_f1 0.22427
wandb: 
wandb: 🚀 View run model1-cnn at: https://wandb.ai/23f3003600s-indian-institute-of-technology-madras/23f3003600-t12026/runs/h5s56ziu
wandb: ⭐️ View project at: https://wandb.ai/23f3003600s-indian-institute-of-technology-madras/23f3003600-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260318_121229-h5s56ziu/logs
